# LeetCode 844 — Backspace String Compare bằng Python

**Link đề:** https://leetcode.com/problems/backspace-string-compare/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String, Stack, Simulation

```python
class Solution:
    def backspaceCompare(self, s: str, t: str) -> bool:
        ...
```

## Mục tiêu học
1. Mô phỏng trình soạn thảo có phím xóa `#` và hiểu vì sao **stack (ngăn xếp)** là cấu trúc tự nhiên cho "xóa ký tự gần nhất".
2. Nắm thao tác stack bằng list Python: `append`, `pop`, kiểm tra rỗng.
3. Giải **follow-up** `O(n)` thời gian, `O(1)` bộ nhớ: **duyệt từ phải sang trái với hai con trỏ** và bộ đếm `skip`.
4. So sánh 4 cách: cắt chuỗi naive → stack → generator duyệt ngược → hai con trỏ.
5. Trace kỹ các ca khó: `#` ở đầu, nhiều `#` liên tiếp, xóa hết, độ dài khác nhau.

## 1. Đề bài và ví dụ

Cho hai chuỗi `s` và `t`. Gõ từng chuỗi vào một trình soạn thảo **rỗng**; ký tự `#` nghĩa là phím **backspace** (xóa ký tự ngay trước con trỏ). Xóa khi văn bản rỗng thì văn bản vẫn rỗng. Trả về `True` nếu hai văn bản cuối cùng bằng nhau.

| s | t | sau khi gõ s | sau khi gõ t | Kết quả |
|---|---|---|---|---|
| `"ab#c"` | `"ad#c"` | `"ac"` | `"ac"` | `True` |
| `"ab##"` | `"c#d#"` | `""` | `""` | `True` |
| `"a#c"` | `"b"` | `"c"` | `"b"` | `False` |

**Constraints:** `1 <= len(s), len(t) <= 200`; chỉ gồm chữ thường và `#`.

**Follow up:** giải được trong `O(n)` thời gian và `O(1)` bộ nhớ không?

In [1]:
# Mô phỏng tay "ab#c" từng phím
text = ""
for ch in "ab#c":
    if ch == "#":
        text = text[:-1]
    else:
        text += ch
    print(f"gõ {ch!r} -> {text!r}")

gõ 'a' -> 'a'
gõ 'b' -> 'ab'
gõ '#' -> 'a'
gõ 'c' -> 'ac'


## 2. Nền tảng Python cần dùng

### 2.1. Stack (ngăn xếp) bằng list

Stack là cấu trúc **vào sau ra trước** (LIFO — Last In, First Out), như chồng đĩa: chỉ thêm/lấy ở **đỉnh**.

- `stack.append(x)`: đặt `x` lên đỉnh — `O(1)`.
- `stack.pop()`: lấy ra và trả về phần tử ở đỉnh — `O(1)`. Gọi trên stack rỗng → `IndexError`.
- `if stack:` — list rỗng được coi là `False`.

Backspace xóa **ký tự gõ gần nhất** = phần tử ở đỉnh stack. Đó là lý do stack khớp hoàn hảo.

In [2]:
stack = []
stack.append("a"); stack.append("b")
print(stack)
print("pop ->", stack.pop(), "| còn", stack)
print("stack rỗng?", not stack, "| bool([]) =", bool([]))
stack.pop()
try:
    stack.pop()
except IndexError as e:
    print("pop trên stack rỗng:", e)

['a', 'b']
pop -> b | còn ['a']
stack rỗng? False | bool([]) = False
pop trên stack rỗng: pop from empty list


### 2.2. Slicing `text[:-1]` — bỏ ký tự cuối

`text[:-1]` lấy từ đầu tới **trước** phần tử cuối. Với chuỗi rỗng, `""[:-1]` vẫn là `""` (không lỗi). Nhưng mỗi lần cắt tạo **chuỗi mới** — tốn `O(độ dài)`.

In [3]:
print(repr("abc"[:-1]), repr("a"[:-1]), repr(""[:-1]))

'ab' '' ''


### 2.3. `"".join(list)` — biến list ký tự thành chuỗi

In [4]:
print("".join(["a", "c"]), repr("".join([])))

ac ''


### 2.4. Generator với `yield`

Hàm có `yield` là **generator**: mỗi lần được hỏi "phần tử tiếp theo", nó chạy tới `yield` kế tiếp rồi **tạm dừng**. Dùng để sinh dần dữ liệu mà không cần lưu tất cả — sẽ dùng ở cách 3.

In [5]:
def count_down(n):
    while n > 0:
        yield n
        n -= 1

g = count_down(3)
print(next(g), next(g), next(g))
print(list(count_down(4)))

3 2 1
[4, 3, 2, 1]


## 3. Cách 1 — Mô phỏng bằng cắt chuỗi (naive)

Làm y như mô phỏng tay: `text = text[:-1]` khi gặp `#`, `text += ch` khi gặp chữ. Rồi so sánh hai kết quả.

Mỗi thao tác có thể tốn `O(n)` vì chuỗi bất biến phải tạo bản mới ⇒ tệ nhất `O(n²)`. Với `n <= 200` vẫn qua, nhưng không đẹp.

In [6]:
def build_slice(s: str) -> str:
    text = ""
    for ch in s:
        if ch == "#":
            text = text[:-1]
        else:
            text += ch
    return text

def compare_slice(s: str, t: str) -> bool:
    return build_slice(s) == build_slice(t)

print(compare_slice("ab#c", "ad#c"), compare_slice("ab##", "c#d#"), compare_slice("a#c", "b"))

True True False


## 4. Cách 2 — Stack: dựng lại văn bản cuối cùng

Duyệt từng ký tự:
- chữ cái → `append` lên stack;
- `#` → nếu stack khác rỗng thì `pop` (xóa ký tự gần nhất), rỗng thì bỏ qua.

Cuối cùng `"".join(stack)` là văn bản. Mỗi ký tự được `append`/`pop` tối đa một lần ⇒ `O(n)`.

In [7]:
def build_stack(s: str) -> str:
    stack = []
    for ch in s:
        if ch != "#":
            stack.append(ch)
        elif stack:            # chỉ pop khi còn ký tự
            stack.pop()
    return "".join(stack)

def compare_stack(s: str, t: str) -> bool:
    return build_stack(s) == build_stack(t)

for s, t in [("ab#c", "ad#c"), ("ab##", "c#d#"), ("a#c", "b")]:
    print(f"{s!r} -> {build_stack(s)!r} | {t!r} -> {build_stack(t)!r} | bằng nhau: {compare_stack(s, t)}")

'ab#c' -> 'ac' | 'ad#c' -> 'ac' | bằng nhau: True
'ab##' -> '' | 'c#d#' -> '' | bằng nhau: True
'a#c' -> 'c' | 'b' -> 'b' | bằng nhau: False


### Trace stack cho `"a##bc#d"`

In [8]:
def trace_stack(s: str) -> str:
    stack = []
    for ch in s:
        if ch != "#":
            stack.append(ch); action = f"append {ch!r}"
        elif stack:
            removed = stack.pop(); action = f"pop {removed!r}"
        else:
            action = "# trên stack rỗng -> bỏ qua"
        print(f"ch={ch!r}: {action:28} stack={stack}")
    return "".join(stack)

print("Kết quả:", repr(trace_stack("a##bc#d")))

ch='a': append 'a'                   stack=['a']
ch='#': pop 'a'                      stack=[]
ch='#': # trên stack rỗng -> bỏ qua  stack=[]
ch='b': append 'b'                   stack=['b']
ch='c': append 'c'                   stack=['b', 'c']
ch='#': pop 'c'                      stack=['b']
ch='d': append 'd'                   stack=['b', 'd']
Kết quả: 'bd'


### Nút thắt của cách 2

Time `O(n + m)` đã tốt, nhưng **bộ nhớ** `O(n + m)` cho hai stack. Follow-up đòi `O(1)` bộ nhớ: không được dựng lại chuỗi.

## 5. Insight — Duyệt **từ phải sang trái** thì biết ngay ký tự nào bị xóa

Đọc từ trái sang phải, khi gặp một chữ ta **chưa biết** nó có bị `#` phía sau xóa không. Nhưng đọc từ **phải sang trái**:

- Gặp `#` → tăng bộ đếm `skip` ("cần bỏ qua thêm 1 chữ phía trái").
- Gặp chữ mà `skip > 0` → chữ này bị xóa: `skip -= 1`, bỏ qua.
- Gặp chữ mà `skip == 0` → chữ này **chắc chắn còn lại** trong văn bản cuối!

Như vậy ta lấy được các ký tự còn lại **theo thứ tự từ cuối lên đầu**, từng ký tự một, không cần lưu. So sánh hai chuỗi từng ký tự như vậy là đủ.

Ví dụ `s = "ab#c"` đọc ngược:

| i | s[i] | skip trước | hành động | skip sau |
|---|---|---|---|---|
| 3 | c | 0 | **giữ 'c'** | 0 |
| 2 | # | 0 | skip += 1 | 1 |
| 1 | b | 1 | bị xóa | 0 |
| 0 | a | 0 | **giữ 'a'** | 0 |

Các ký tự giữ lại theo thứ tự ngược: `c, a` → văn bản `"ac"` ✓.

## 6. Cách 3 — Generator duyệt ngược (cầu nối dễ hiểu)

Viết generator `kept_chars_reversed(s)` sinh lần lượt các ký tự còn lại từ cuối về đầu, rồi so sánh hai dãy. Dùng `itertools.zip_longest` để ghép hai dãy có thể dài khác nhau (phần thiếu được điền `None`).

In [9]:
from itertools import zip_longest

def kept_chars_reversed(s: str):
    skip = 0
    for ch in reversed(s):
        if ch == "#":
            skip += 1
        elif skip > 0:
            skip -= 1
        else:
            yield ch

print(list(kept_chars_reversed("ab#c")), list(kept_chars_reversed("a##bc#d")))
print(list(zip_longest("ab", "abc")))

def compare_generator(s: str, t: str) -> bool:
    return all(x == y for x, y in zip_longest(kept_chars_reversed(s), kept_chars_reversed(t)))

print(compare_generator("ab#c", "ad#c"), compare_generator("ab##", "c#d#"), compare_generator("a#c", "b"))

['c', 'a'] ['d', 'b']
[('a', 'a'), ('b', 'b'), (None, 'c')]
True True False


`all(...)` trả `True` nếu **mọi** phần tử đúng và dừng ngay khi gặp phần tử sai đầu tiên. Bộ nhớ của generator chỉ là vài biến ⇒ `O(1)`.

## 7. Lời giải chính — hai con trỏ từ phải sang trái (từng đoạn rồi ghép)

Viết tường minh, không dùng generator, để thấy rõ mọi trạng thái.

**Đoạn 1 — hàm phụ:** `next_valid(st, i)` trả về chỉ số của ký tự **còn lại** gần nhất tại hoặc bên trái `i` (hoặc `-1` nếu hết).

```python
def next_valid(st, i):
    skip = 0
    while i >= 0:
        if st[i] == "#":
            skip += 1
        elif skip > 0:
            skip -= 1
        else:
            return i          # ký tự này còn lại
        i -= 1
    return -1
```

**Đoạn 2 — vòng chính:** `i = len(s) - 1`, `j = len(t) - 1`. Lặp:
1. `i = next_valid(s, i)`, `j = next_valid(t, j)`.
2. Nếu cả hai `-1` → hai văn bản hết cùng lúc → `True`.
3. Nếu chỉ một bên `-1` → độ dài khác nhau → `False`.
4. Nếu `s[i] != t[j]` → `False`.
5. Ngược lại, lùi cả hai: `i -= 1`, `j -= 1`.

In [10]:
def next_valid(st: str, i: int) -> int:
    skip = 0
    while i >= 0:
        if st[i] == "#":
            skip += 1
        elif skip > 0:
            skip -= 1
        else:
            return i
        i -= 1
    return -1

def compare_two_pointers(s: str, t: str) -> bool:
    i, j = len(s) - 1, len(t) - 1
    while True:
        i = next_valid(s, i)
        j = next_valid(t, j)
        if i < 0 and j < 0:
            return True
        if i < 0 or j < 0:
            return False
        if s[i] != t[j]:
            return False
        i -= 1
        j -= 1

print(compare_two_pointers("ab#c", "ad#c"), compare_two_pointers("ab##", "c#d#"), compare_two_pointers("a#c", "b"))

True True False


## 8. TRACE từng bước

### 8.1. Bảng trace `s = "bxj##tw"`, `t = "bxo#j##tw"` (cả hai thành `"btw"`)

| vòng | i (s[i]) còn lại | j (t[j]) còn lại | so sánh | hành động |
|---|---|---|---|---|
| 1 | 6 ('w') | 8 ('w') | bằng | i=5, j=7 |
| 2 | 5 ('t') | 7 ('t') | bằng | i=4, j=6 |
| 3 | s[4]=#,s[3]=# → skip 2; s[2]='j',s[1]='x' bị xóa → 0 ('b') | t[6]=#,t[5]=# → skip 2; t[4]='j' xóa; t[3]=# skip→2; t[2]='o', t[1]='x' xóa → 0 ('b') | bằng | i=−1, j=−1 |
| 4 | −1 | −1 | — | cả hai hết → `True` |

In [11]:
def trace_two_pointers(s: str, t: str) -> bool:
    i, j = len(s) - 1, len(t) - 1
    rnd = 1
    print(f"s={s!r}, t={t!r}")
    while True:
        i = next_valid(s, i)
        j = next_valid(t, j)
        si = s[i] if i >= 0 else "∅"
        tj = t[j] if j >= 0 else "∅"
        print(f"vòng {rnd}: i={i:>2} ({si}) | j={j:>2} ({tj})", end="  ")
        if i < 0 and j < 0:
            print("-> cả hai hết: True"); return True
        if i < 0 or j < 0:
            print("-> một bên hết trước: False"); return False
        if s[i] != t[j]:
            print("-> khác nhau: False"); return False
        print("-> bằng nhau, lùi cả hai")
        i -= 1; j -= 1; rnd += 1

trace_two_pointers("bxj##tw", "bxo#j##tw");

s='bxj##tw', t='bxo#j##tw'
vòng 1: i= 6 (w) | j= 8 (w)  -> bằng nhau, lùi cả hai
vòng 2: i= 5 (t) | j= 7 (t)  -> bằng nhau, lùi cả hai
vòng 3: i= 0 (b) | j= 0 (b)  -> bằng nhau, lùi cả hai
vòng 4: i=-1 (∅) | j=-1 (∅)  -> cả hai hết: True


### 8.2. Trace ca khác nhau và ca một bên hết trước

In [12]:
trace_two_pointers("a#c", "b");
print()
trace_two_pointers("xy#", "x#");

s='a#c', t='b'
vòng 1: i= 2 (c) | j= 0 (b)  -> khác nhau: False

s='xy#', t='x#'
vòng 1: i= 0 (x) | j=-1 (∅)  -> một bên hết trước: False


## 9. Code nộp LeetCode

Bản hai con trỏ `O(1)` bộ nhớ (đáp ứng follow-up), viết gọn trong một method:

In [13]:
class Solution:
    def backspaceCompare(self, s: str, t: str) -> bool:
        def next_valid(st: str, i: int) -> int:
            skip = 0
            while i >= 0:
                if st[i] == "#":
                    skip += 1
                elif skip > 0:
                    skip -= 1
                else:
                    return i
                i -= 1
            return -1

        i, j = len(s) - 1, len(t) - 1
        while True:
            i, j = next_valid(s, i), next_valid(t, j)
            if i < 0 or j < 0:
                return i < 0 and j < 0       # True chỉ khi cả hai cùng hết
            if s[i] != t[j]:
                return False
            i -= 1
            j -= 1

sol = Solution()
assert sol.backspaceCompare("ab#c", "ad#c") is True
assert sol.backspaceCompare("ab##", "c#d#") is True
assert sol.backspaceCompare("a#c", "b") is False
print("Solution PASS 3 ví dụ đề")

Solution PASS 3 ví dụ đề


Bản stack — ngắn, dễ nhớ, cũng được chấp nhận:

In [14]:
class SolutionStack:
    def backspaceCompare(self, s: str, t: str) -> bool:
        def build(x: str) -> list:
            st = []
            for ch in x:
                if ch != "#":
                    st.append(ch)
                elif st:
                    st.pop()
            return st
        return build(s) == build(t)

print(SolutionStack().backspaceCompare("ab#c", "ad#c"))

True


## 10. Vì sao đúng? Độ phức tạp

**Stack:** invariant — sau khi xử lý tiền tố `s[:k]`, stack bằng đúng văn bản trên màn hình sau `k` phím. Chữ → thêm vào cuối; `#` → xóa ký tự cuối nếu có. Đó đúng là định nghĩa của trình soạn thảo.

**Hai con trỏ:** Một chữ ở vị trí `p` **còn lại** trong văn bản cuối ⇔ trong đoạn `s[p+1:]`, số `#` "chưa dùng" để xóa các chữ nằm giữa không đủ để chạm tới `p`. Đọc từ phải sang trái với bộ đếm `skip` kiểm tra đúng điều này: `skip` = số backspace còn "nợ" phải xóa về phía trái. Do đó `next_valid` trả về đúng các ký tự còn lại **theo thứ tự từ cuối lên**. Hai văn bản bằng nhau ⇔ hai dãy này bằng nhau phần tử-theo-phần tử và kết thúc cùng lúc — đúng là điều vòng lặp kiểm tra.

**Độ phức tạp:** `n = len(s)`, `m = len(t)`. Con trỏ `i` chỉ giảm (cả trong `next_valid` lẫn vòng chính), tương tự `j` ⇒ tổng `O(n + m)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Cắt chuỗi | `O(n² + m²)` tệ nhất | `O(n + m)` |
| 2. Stack | `O(n + m)` | `O(n + m)` |
| 3. Generator ngược | `O(n + m)` | `O(1)` |
| 4. Hai con trỏ | `O(n + m)` | `O(1)` — **đạt follow-up** |

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. `#` ở **đầu** chuỗi (`"#a"`), nhiều `#` hơn số chữ (`"a###"`).
3. Xóa hết một bên, bên kia còn chữ.
4. Độ dài văn bản cuối khác nhau nhưng một bên là "tiền tố" bên kia (`"xy#"` vs `"x#"`... → `"x"` vs `""`).
5. `#` xen kẽ phức tạp.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.backspaceCompare("ab#c", "ad#c"), True)
check("đề 2", sol.backspaceCompare("ab##", "c#d#"), True)
check("đề 3", sol.backspaceCompare("a#c", "b"), False)
check("# ở đầu", sol.backspaceCompare("#a", "a"), True)
check("thừa #", sol.backspaceCompare("a###", "#"), True)
check("một bên rỗng", sol.backspaceCompare("a#", "b"), False)
check("tiền tố", sol.backspaceCompare("xy#", "x#"), False)
check("phức tạp", sol.backspaceCompare("bxj##tw", "bxo#j##tw"), True)
check("phức tạp 2", sol.backspaceCompare("bbbextm", "bbb#extm"), False)
check("không #", sol.backspaceCompare("abc", "abc"), True)

PASS đề 1: True
PASS đề 2: True
PASS đề 3: False
PASS # ở đầu: True
PASS thừa #: True
PASS một bên rỗng: False
PASS tiền tố: False
PASS phức tạp: True
PASS phức tạp 2: False
PASS không #: True


### 11.1. Đối chiếu 6 cài đặt trên chuỗi ngẫu nhiên (có seed)

Dùng bảng ký tự nhỏ `"ab#"` để có nhiều `#` và nhiều ca bằng nhau.

In [16]:
import random
random.seed(844)
impls = [compare_slice, compare_stack, compare_generator, compare_two_pointers,
         Solution().backspaceCompare, SolutionStack().backspaceCompare]
equal_count = 0
for _ in range(3000):
    s = "".join(random.choice("ab#") for _ in range(random.randint(1, 8)))
    t = "".join(random.choice("ab#") for _ in range(random.randint(1, 8)))
    expected = build_slice(s) == build_slice(t)
    equal_count += expected
    for f in impls:
        assert f(s, t) == expected, (s, t, f)
print(f"3000 cặp ngẫu nhiên: 6 cài đặt khớp nhau ({equal_count} cặp bằng nhau)")

3000 cặp ngẫu nhiên: 6 cài đặt khớp nhau (235 cặp bằng nhau)


## 12. Lỗi người mới hay gặp

### Lỗi 1: `pop()` khi stack rỗng

```python
if ch == "#":
    stack.pop()        # "#a" -> IndexError ngay ký tự đầu
```

In [17]:
def wrong_pop(s):
    stack = []
    for ch in s:
        if ch == "#":
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

try:
    wrong_pop("#a")
except IndexError as e:
    print("Lỗi:", e, "-> phải viết `elif stack: stack.pop()`")
print("đúng:", repr(build_stack("#a")))

Lỗi: pop from empty list -> phải viết `elif stack: stack.pop()`
đúng: 'a'


### Lỗi 2: duyệt **trái → phải** với bộ đếm `skip`

Nhìn chữ trước rồi mới thấy `#` phía sau → không biết chữ nào bị xóa. Một cách sai điển hình là "đếm số `#` rồi bỏ đi bấy nhiêu chữ cuối":

In [18]:
def wrong_count(s):
    letters = [ch for ch in s if ch != "#"]
    k = s.count("#")
    return "".join(letters[:max(0, len(letters) - k)])

print("sai :", repr(wrong_count("#ab")), "| đúng:", repr(build_stack("#ab")))
print("sai :", repr(wrong_count("a#b")), "| đúng:", repr(build_stack("a#b")))

sai : 'a' | đúng: 'ab'
sai : 'a' | đúng: 'b'


Vì sao sai: `#` ở đầu (văn bản rỗng) **không** xóa gì, nhưng cách đếm vẫn trừ đi. Và `#` chỉ xóa ký tự **trước nó**, không xóa chữ gõ sau.

### Lỗi 3: hai con trỏ nhưng chỉ kiểm tra `s[i] != t[j]`, quên trường hợp một bên hết trước

In [19]:
def wrong_end(s, t):
    i, j = len(s) - 1, len(t) - 1
    while True:
        i, j = next_valid(s, i), next_valid(t, j)
        if i < 0 and j < 0:
            return True
        if s[i] != t[j]:              # i = -1 -> s[-1] là ký tự CUỐI, không báo lỗi!
            return False
        i -= 1; j -= 1

print("sai :", wrong_end("a", "aa"), "| đúng:", Solution().backspaceCompare("a", "aa"))

sai : True | đúng: False


Ca `"a"` vs `"aa"`: sau khi so `a` với `a`, `i = -1` nhưng `j = 0`. Code sai đọc `s[-1]` (= `'a'`, ký tự cuối do chỉ số âm của Python!) → so bằng nhau → kết luận nhầm `True`. Chỉ số âm của Python khiến lỗi này **không báo** `IndexError` — rất khó phát hiện.

### Lỗi 4: quên reset `skip` cho mỗi lần tìm (nếu viết `skip` dùng chung hai chuỗi)

Mỗi chuỗi cần **bộ đếm riêng** `skip_s`, `skip_t`. Dùng chung một biến sẽ để `#` của `s` xóa nhầm chữ của `t`.

## 13. Mẹo nhận dạng pattern

- "Xóa / hủy / ghép cặp với phần tử **gần nhất trước đó**" → **stack**: *Valid Parentheses (20)*, *Remove All Adjacent Duplicates (1047)*, *Make The String Great (1544)*, *Crawler Log Folder (1598)*.
- Follow-up `O(1)` bộ nhớ cho bài stack "xóa phía trước" → thử **duyệt ngược** với bộ đếm (thông tin về tương lai trở thành quá khứ).
- So sánh hai dãy được sinh dần → **hai con trỏ** chạy song song, kết thúc phải **cùng lúc**.
- Cẩn thận chỉ số âm trong Python: `s[-1]` hợp lệ, không ném lỗi.

## 14. Biến thể

### 14.1. Thêm phím "xóa cả từ" `@` (xóa tới dấu cách trước đó)

In [20]:
def build_with_word_delete(s: str) -> str:
    stack = []
    for ch in s:
        if ch == "#":
            if stack:
                stack.pop()
        elif ch == "@":
            while stack and stack[-1] != " ":
                stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

print(repr(build_with_word_delete("hi there@you")))

'hi you'


`stack[-1]` là phần tử ở **đỉnh** (xem mà không lấy ra).

### 14.2. Remove All Adjacent Duplicates (LeetCode 1047)

Hai ký tự **kề nhau giống nhau** thì "triệt tiêu". Cùng khung stack: nếu đỉnh bằng ký tự mới thì `pop`, ngược lại `append`.

In [21]:
def remove_adjacent_duplicates(s: str) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

print(remove_adjacent_duplicates("abbaca"), remove_adjacent_duplicates("azxxzy"))

ca ay


### 14.3. Đếm số ký tự còn lại mà không dựng chuỗi

Chỉ cần một biến `length`: chữ → `+1`, `#` → `max(0, length - 1)`.

In [22]:
def final_length(s: str) -> int:
    length = 0
    for ch in s:
        length = length + 1 if ch != "#" else max(0, length - 1)
    return length

print(final_length("ab#c"), final_length("a###"), final_length("bxo#j##tw"))

2 0 3


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Văn bản cuối của `"y#fo##f"` và `"y#f#o##f"` là gì? Hai chuỗi có bằng nhau không?

### Bài 2 — Dự đoán output
Không chạy, đoán: `""[:-1]`, `bool([])`, `"abc"[-1]`, `[1, 2, 3].pop()`.

### Bài 3 — Điền TODO cho hàm `next_valid`

In [23]:
def ex3_next_valid(st: str, i: int) -> int:
    skip = 0
    while i >= 0:
        # TODO: 3 nhánh: gặp '#', gặp chữ khi skip > 0, gặp chữ khi skip == 0
        pass
    return -1

# assert ex3_next_valid("ab#c", 3) == 3
# assert ex3_next_valid("ab#c", 2) == 0
# assert ex3_next_valid("a#", 1) == -1

### Bài 4 — Remove All Adjacent Duplicates (1047)
Tự viết lại hàm ở mục 14.2 mà không nhìn.

In [24]:
def ex4_remove_duplicates(s: str) -> str:
    # TODO: stack
    pass

# assert ex4_remove_duplicates("abbaca") == "ca"

### Bài 5 — Crawler Log Folder (1598)
Cho list thao tác thư mục: `"../"` (lên thư mục cha, nếu đang ở gốc thì đứng yên), `"./"` (đứng yên), `"x/"` (vào thư mục con). Trả về số bước tối thiểu để quay về gốc.

In [25]:
def ex5_min_operations(logs: list[str]) -> int:
    # TODO: chỉ cần một biến depth (giống bài 14.3)
    pass

# assert ex5_min_operations(["d1/", "d2/", "../", "d21/", "./"]) == 2
# assert ex5_min_operations(["d1/", "../", "../", "../"]) == 0

### Bài 6 — Mini challenge: văn bản cuối dài nhất?
Cho list các chuỗi có `#`. Trả về chuỗi gốc có **văn bản cuối dài nhất** (hòa thì lấy chuỗi xuất hiện trước). Yêu cầu: `O(1)` bộ nhớ phụ cho mỗi chuỗi (không dựng stack).

In [26]:
def ex6_longest_final(strs: list[str]) -> str:
    # TODO
    pass

# assert ex6_longest_final(["ab#c", "abc", "a###"]) == "abc"

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: dùng stack, gõ từng phím.
- Bài 5: độ sâu thư mục giống "độ dài văn bản"; `"../"` giống `#`.
- Bài 6: chỉ cần **độ dài** văn bản cuối (mục 14.3).

**Mức 2 — cú pháp**
- Bài 3: `if st[i] == "#": skip += 1` / `elif skip > 0: skip -= 1` / `else: return i`; cuối vòng `i -= 1`.
- Bài 4: `if stack and stack[-1] == ch: stack.pop() else: stack.append(ch)`.

**Mức 3 — gần code**
- Bài 5: `for op in logs: if op == "../": depth = max(0, depth - 1) elif op != "./": depth += 1`.
- Bài 6: `max(strs, key=final_length)` — `max` với `key` trả phần tử **đầu tiên** đạt giá trị lớn nhất.

## 17. Đáp án tham khảo

**Bài 1:** `"y#fo##f"`: y → xóa → f → fo → f → (rỗng) → f ⇒ `"f"`. `"y#f#o##f"`: y → xóa → f → xóa → o → xóa → (rỗng, # thừa) → f ⇒ `"f"`. Bằng nhau ⇒ `True`.

**Bài 2:** `""`, `False`, `"c"`, `3`.

In [27]:
print(repr(""[:-1]), bool([]), "abc"[-1], [1, 2, 3].pop())
assert build_stack("y#fo##f") == "f" and build_stack("y#f#o##f") == "f"
assert Solution().backspaceCompare("y#fo##f", "y#f#o##f") is True

def ans3_next_valid(st, i):
    skip = 0
    while i >= 0:
        if st[i] == "#":
            skip += 1
        elif skip > 0:
            skip -= 1
        else:
            return i
        i -= 1
    return -1

def ans4_remove_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def ans5_min_operations(logs):
    depth = 0
    for op in logs:
        if op == "../":
            depth = max(0, depth - 1)
        elif op != "./":
            depth += 1
    return depth

def ans6_longest_final(strs):
    return max(strs, key=final_length)

assert ans3_next_valid("ab#c", 3) == 3
assert ans3_next_valid("ab#c", 2) == 0
assert ans3_next_valid("a#", 1) == -1
assert ans4_remove_duplicates("abbaca") == "ca"
assert ans5_min_operations(["d1/", "d2/", "../", "d21/", "./"]) == 2
assert ans5_min_operations(["d1/", "../", "../", "../"]) == 0
assert ans6_longest_final(["ab#c", "abc", "a###"]) == "abc"
print("Đáp án bài 1-6 PASS")

'' False c 3
Đáp án bài 1-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

**Template stack (dễ nhớ):**
```python
st = []
for ch in s:
    if ch != "#": st.append(ch)
    elif st:      st.pop()
```

**Template hai con trỏ `O(1)` bộ nhớ:**
```python
def next_valid(st, i):
    skip = 0
    while i >= 0:
        if st[i] == "#": skip += 1
        elif skip:       skip -= 1
        else:            return i
        i -= 1
    return -1
# vòng chính: lấy next_valid hai bên; cả hai -1 -> True; một bên -1 -> False; khác ký tự -> False; lùi cả hai
```

**Checklist trước khi submit:**
- [ ] `pop` chỉ khi stack **khác rỗng**?
- [ ] Hai con trỏ: xử lý ca **một bên hết trước** (đừng để `s[-1]` lừa)?
- [ ] Mỗi chuỗi có `skip` riêng, reset mỗi lần gọi `next_valid`?
- [ ] Sau khi so bằng, lùi **cả hai** con trỏ?
- [ ] Đã test `#` ở đầu, thừa `#`, xóa hết, độ dài khác nhau?